In [4]:
import math
import numpy as np

import sys

import plotly.graph_objects as go

import matplotlib
import matplotlib.pyplot as plt
import mplhep as hep

from h5flow.data import dereference
import h5flow

from h5flow.data import dereference
import h5flow

import numpy as np

import boost_histogram as bh

import hist

import uproot

import os

from scipy.spatial.distance import pdist, squareform, cdist

import awkward as ak

from sklearn.neighbors import KDTree

from sklearn.cluster import DBSCAN

# everything in iminuit is done through the Minuit object, so we import it
from iminuit import Minuit

# we also need a cost function to fit and import the LeastSquares function
from iminuit.cost import LeastSquares

# display iminuit version
import iminuit

from sklearn.linear_model import LinearRegression

from hist import Hist

import scipy

import numpy.lib.recfunctions as rfn

In [2]:
# ??DBSCAN

In [5]:
plt.ioff()

In [6]:
def plot_three_views(hits, axs=None, **kwargs):
    #if axs is None:
    #    fig, axs = plt.subplots(2, 2, figsize=(5*2, 5*2))
    #    assert fig == axs[0,0].get_figure()
    #elif isinstance(axs, np.ndarray):
    #    if axs.shape == (2, 2):
    #        fig = axs[0,0].get_figure()
    #    else:
    #        raise TypeError('Axes passed in are not 2x2')
    #else:
    #    raise TypeError('axs is not properly given')
    
    # ridx, cidx
    axis_dict = {
        # y vs. x
        (0, 0) : { 'label' : {'x' : 'x [cm]', 'y' : 'y [cm]'},
                    'key' : {'x' : 'x', 'y' : 'y'}},
        # (0, 1) : # empty
        (1, 0) : { 'label' : {'x' : 'x [cm]', 'y' : 'z [cm]'},
                  'key' : {'x' : 'x', 'y' : 'z'}},
        (1, 1) : { 'label' : {'x' : 'y [cm]', 'y' : 'z [cm]'},
                  'key' : {'x' : 'y', 'y' : 'z'}}
    }
    all_hits = kwargs.get('all_hits', None)
    if isinstance(all_hits, np.ndarray):
        for k, v in axis_dict.items():
            axs[k[0], k[1]].scatter(all_hits[v['key']['x']], all_hits[v['key']['y']], label='all hits')
    for k, v in axis_dict.items():
        axs[k[0], k[1]].scatter(hits[v['key']['x']], hits[v['key']['y']], label=kwargs.get('label', 'selected'))

    for k, v in axis_dict.items():
        axs[k[0], k[1]].set_xlabel(v['label']['x'])
        axs[k[0], k[1]].set_ylabel(v['label']['y'])
        axs[k[0], k[1]].legend()
    
    # return fig

In [7]:
def line1D(X, y):
    reg = LinearRegression().fit(X.reshape(-1, 1), y)
    return reg.coef_[0], reg.intercept_

def yz_line(hits):
    '''
    z = ky + b
    dz = kdy
    return k, b, (1/sqrt(1+k**2), k/sqrt(1+k**2))
    tagent vector = (1/sqrt(1+k**2), k/sqrt(1+k**2))
    '''
    # reg = LinearRegression().fit(hits['y'].reshape(-1, 1), hits['z'])
    # k = reg.coef_[0]
    # b = reg.intercept_
    k, b = line1D(hits['y'], hits['z'])
    return k, b, (1/np.sqrt(1+k**2), k/np.sqrt(1+k**2))
def proj_yz(hits, k_yz, b_yz):
    '''
    z = ky + b
    '''
    tg_yz = np.array([1/np.sqrt(1+k_yz**2), k_yz/np.sqrt(1+k_yz**2)])
    o_yz = (0, b_yz)
    points_yz = np.column_stack([hits['y'], hits['z']]) - o_yz
    d_tg = np.dot(points_yz, tg_yz)
    # for i in range(10):
    #     print(tg_yz[0]* points_yz[i][0] + tg_yz[1]* points_yz[i][1] - d_tg[i], points_yz[i], rock_muon_hits['y'][i], rock_muon_hits['z'][i], o_yz)
    d_nm = np.cross(points_yz, tg_yz)
    # for i in range(10):
    #     print(-tg_yz[0]* points_yz[i][1] + tg_yz[1]* points_yz[i][0] - d_nm[i], points_yz[i], rock_muon_hits['y'][i], rock_muon_hits['z'][i], o_yz, tg_yz)
    return d_tg, d_nm
def xline(hits, reflabel='d_tg'):
    '''
    x = k* reflabel + b
    '''
    k, b = line1D(hits[reflabel], hits['x'])
    return k, b, (1/np.sqrt(1+k**2), k/np.sqrt(1+k**2))

In [8]:
def plot_line(k, b, ax, **kwargs):
    x0, x1 = ax.get_xlim()
    x = np.arange(x0, x1, step=0.01)
    y = k * x + b
    ax.plot(x, y, label='line fit', c=kwargs.get('color'))
    ax.legend()

In [9]:
# not validated
def sel_uni_pxl(hits, att='Q'):
    points_yz = np.column_stack([hits['io_group'], hits['io_channel'], hits['y'], hits['z']])
    clustering = DBSCAN(eps=0.0001, min_samples=1).fit(points_yz)
    selected_hit = []
    unique_labels = np.unique(clustering.labels_)
    totQ = np.zeros(hits.shape[0], dtype=float)
    totN = np.zeros(hits.shape[0], dtype=int)
    totQ_cp = totQ.copy()
    totN_cp = totN.copy()
    accQ = np.zeros(hits.shape[0], dtype=float)
    avg_i = np.zeros(hits.shape[0], dtype=float)
    dt = np.zeros(hits.shape[0], dtype=float)
    
    dist = pdist((points_yz - clustering.components_).reshape(-1, 1))
    # if np.sum(dist)>1E-6:
    #     print(np.sum(dist))
    # print(np.sum(dist))
    print('negative Q', hits['Q'][hits['Q']<0])
    
    for ilabel, unique_label in enumerate(unique_labels):
        m = clustering.labels_ == unique_label
        idxs = np.asarray(m).nonzero()[0]
        d = hits[m]
        selected_hit.append(d[np.argmax(d[att])])
        totQ[ilabel] = np.sum(d['Q'])
        totN[ilabel] = len(d)
        totQ_cp[m] = np.sum(d['Q'])
        totN_cp[m] = len(d)
        sorted_indices = np.argsort(d['t_drift'])
        dt[m] = hits['t_drift'][m] - selected_hit[-1]['t_drift']
        q = 0
        for i in range(len(sorted_indices)):
            q += d['Q'][sorted_indices[i]]
            accQ[idxs[sorted_indices[i]]] = np.float_(q)
            if i == 0:
                avg_i[idxs[sorted_indices[i]]] = -1E9
            else:
                avg_i[idxs[sorted_indices[i]]] = d['Q'][sorted_indices[i]] / (d['t_drift'][sorted_indices[i]] - d['t_drift'][sorted_indices[i-1]])
            # if len(d) > 1:
            #     print(q, accQ[m])
        # print(d['t_drift'], d['Q'], avg_i[m])
        # if len(d) > 1:
        #   print(accQ[m], totQ_cp[m], d['t_drift'], q)
        #   print(d['t_drift'], dt[m])
    uni_pxl = np.array(selected_hit, dtype=hits.dtype)
    totQ = totQ[:len(uni_pxl)]
    totN = totN[:len(uni_pxl)]
    uni_pxl = rfn.append_fields(uni_pxl, names=['totQ', 'totN'], data=[totQ, totN], usemask=False)
    extended_hits = rfn.append_fields(hits, names=['totQ', 'totN'], data=[totQ_cp, totN_cp], usemask=False)
    extended_hits = rfn.append_fields(extended_hits, names=['accQ', 'dt', 'avg_i'], data=[accQ, dt, avg_i], usemask=False)
    return uni_pxl, extended_hits

In [10]:
def prepare_tracks(f_name):
    f = h5flow.data.H5FlowDataManager(f_name, 'r')
    tracks2hits = dereference(
        f['/analysis/rock_muon_tracks/data']['rock_muon_id'],     # indices of A to load references for, shape: (n,)
        f['/analysis/rock_muon_tracks/ref/charge/calib_prompt_hits/ref'],  # references to use, shape: (L,)
        f['/charge/calib_prompt_hits/data'],
        ref_direction = (0,1)# dataset to load, shape: (M,)
    )
    tracks = f['/analysis/rock_muon_tracks/data']
    
    return tracks, tracks2hits

def analyze_hits_per_track(tracks, tracks2hits, itrk, att='t_drift'):
    track = tracks[itrk]
    t2h = tracks2hits[itrk]
    rock_muon_hits = np.array([tup for tup in t2h if not any(tup.mask)], dtype = t2h.dtype)
    
    k_yz, b_yz, tg_yz = yz_line(rock_muon_hits)
        
    d_tg, d_nm = proj_yz(rock_muon_hits, k_yz, b_yz)
        
    extended_hits = rfn.append_fields(rock_muon_hits, names=('d_tg', 'd_nm'), data=(d_tg, d_nm), usemask=False)
        
    uni_pxls, extended_hits = sel_uni_pxl(extended_hits, att=att)
    
    reflabel = 'y'
    refaxislabel = 'y [cm]'
    k_x, b_x, _ = xline(uni_pxls, reflabel='y')
    
    ref_x = k_x * extended_hits[reflabel] + b_x
    dx = extended_hits['x'] - ref_x
    sign = np.where(extended_hits['io_group'] % 2, np.ones(extended_hits.shape[0]), -1*np.ones(extended_hits.shape[0]))
    dx = dx * sign
    extended_hits = rfn.append_fields(extended_hits, names='dx', data=dx, usemask=False)

    
    ### plotting

    fig, axs = plt.subplots(3, 2, figsize=(5*2, 5*3))
    
    plot_three_views(uni_pxls, axs, all_hits=extended_hits)
    axs[0,1].scatter(extended_hits[reflabel], extended_hits['x'], label='all hits')
    axs[0,1].scatter(uni_pxls[reflabel], uni_pxls['x'], label='selected')
    axs[0,1].set_xlabel(refaxislabel)
    axs[0,1].set_ylabel('x [cm]')
    
    plot_line(ax=axs[1,1], k=k_yz, b=b_yz, color='g')
    plot_line(ax=axs[0,1], k=k_x, b=b_x, color='g')
    
    # ax=
    sc0 = axs[2,0].scatter(uni_pxls['y'], uni_pxls['z'], c=uni_pxls['totQ'], s=1,
                    norm=matplotlib.colors.LogNorm(), cmap='rainbow')
    sc1 = axs[2,1].scatter(uni_pxls['y'], uni_pxls['z'], c=uni_pxls['totN'], s=1,
                    norm=matplotlib.colors.BoundaryNorm(boundaries=np.arange(0, 5, 1), ncolors=256),
                          cmap='RdBu_r')
    fig.colorbar(sc0,
             ax=axs[2,0], orientation='vertical', label='total charge at pixel')
    fig.colorbar(sc1,
             ax=axs[2,1], orientation='vertical', label='N hits at pixel', extend='max')
    
    # display(fig)
    
    return extended_hits, fig

In [47]:
def analysis(hits):
    hdx = (
        Hist.new.Regular(60, -1.5, 1.5, name='x', label='dx [cm]')
        .Double()
    )
    hyz = (
        Hist.new.Regular(60, -1.5, 1.5, name='x', label='tangent on yz [cm]')
        .Double()
    )
    hyz_perp = (
        Hist.new.Regular(60, -1.5, 1.5, name='x', label='perp. to tangent on yz [cm]')
        .Double()
    )
    hdx_vs_totQ = (
        Hist.new.Regular(50, 0, 100, name='x', label='totQ at a pixel [ke-]')
        .Regular(60, -1.5, 1.5, name='y', label='dx [cm]')
        .Double()
    )
    
    hdx_vs_Q = (
        Hist.new
        .Regular(60, 0, 100, name='x', label='Q [ke]')
        .Regular(60, -1.5, 1.5, name='y', label='dx [cm]')
        .Double()
    )
    
    hdx_vs_Q_n = (
        Hist.new
        .Regular(60, 0, 100, name='x', label='Q [ke]')
        .Regular(60, -1.5, 1.5, name='y', label='dx [cm]')
        .Double()
    )
    haccQ_vs_dx = (
        Hist.new.Regular(100, -2.5, 2.5, name='x', label='dx [cm]')
        .Regular(60, 0, 1.2, name='y', label='fractional accQ at a pixel')
        .Double()
    )
    
    haccQ_vs_dt = (
        Hist.new.Regular(100, -100, 50, name='x', label='dt [0.1us]')
        .Regular(60, 0, 1.2, name='y', label='fractional accQ at a pixel')
        .Double()
    )
    
    havg_i_vs_dx = (
        Hist.new.Regular(100, -2.5, 2.5, name='x', label='dx [cm]')
        .Regular(40, 0, 1, name='y', label='average current at a pixel [ke-/0.1us]')
        .Double()
    )
    
    hfrac_avg_i_vs_dx = (
        Hist.new.Regular(100, -2.5, 2.5, name='x', label='dx [cm]')
        .Regular(40, 0, 0.04, name='y', label='fractional average current at a pixel [/0.1us]')
        .Double()
    )
    
    hdx_vs_totN = (
        Hist.new.Regular(6, -0.5, 5.5, name='x', label='N hit at a pixel')
        .Regular(60, -1.5, 1.5, name='y', label='dx [cm]')
        .Double()
    )
    
    hdx.fill(hits['dx'])
    hdx_vs_totQ.fill(
        hits['totQ'], hits['dx']
    )
    hdx_vs_totN.fill(
        hits['totN'], hits['dx']
    )
    hdx_vs_Q.fill(hits['Q'], hits['dx'])
    hyz_perp.fill(hits['d_nm'])
    
    dx_multiple_n = hits['dx'][hits['totN'] > 1]
    accQ_multiple_n = hits['accQ'][hits['totN'] > 1]
    totQ_multiple_n = hits['totQ'][hits['totN'] > 1]
    avg_i_multiple_n = hits['avg_i'][hits['totN'] > 1]
    # dx_multiple_n = hits['dx']
    # accQ_multiple_n = hits['accQ']
    # totQ_multiple_n = hits['totQ']
    
    haccQ_vs_dx.fill(dx_multiple_n, accQ_multiple_n/totQ_multiple_n)
    haccQ_vs_dt.fill(hits['dt'][hits['totN'] > 1], accQ_multiple_n/totQ_multiple_n)
    
    hfrac_avg_i_vs_dx.fill(dx_multiple_n, avg_i_multiple_n/totQ_multiple_n)
    havg_i_vs_dx.fill(dx_multiple_n, avg_i_multiple_n)
    
    hdx_vs_Q_n.fill(hits['Q'][hits['totN'] > 1], dx_multiple_n)
    
    multi_hits = hits[hits['totN'] > 1]
    print(multi_hits['accQ'][accQ_multiple_n/totQ_multiple_n > 1], 
         multi_hits['totQ'][accQ_multiple_n/totQ_multiple_n > 1],
          multi_hits['dt'][accQ_multiple_n/totQ_multiple_n > 1],
          multi_hits['Q'][accQ_multiple_n/totQ_multiple_n > 1],
          multi_hits[accQ_multiple_n/totQ_multiple_n > 1]
         )
    
    fig, axs = plt.subplots(5, 2, figsize=(5*2, 5*5)
                           )
    hep.histplot(hdx, ax=axs[0,0])
    hep.histplot(hyz_perp, ax=axs[0,1])
    hep.hist2dplot(hdx_vs_totQ, ax=axs[1,0], cmap='rainbow', cmin=0.0000001)
    hep.hist2dplot(hdx_vs_totN, ax=axs[1,1], cmap='rainbow', cmin=0.0000001)
    hep.hist2dplot(haccQ_vs_dx, ax=axs[2,0], cmap='rainbow', cmin=0.0000001)
    hep.hist2dplot(haccQ_vs_dt, ax=axs[2,1], cmap='rainbow', cmin=0.0000001)
    hep.hist2dplot(hfrac_avg_i_vs_dx, ax=axs[3,0], cmap='rainbow', cmin=0.00000000001)
    hep.hist2dplot(havg_i_vs_dx, ax=axs[3,1], cmap='rainbow', cmin=0.00000000001)
    hep.hist2dplot(hdx_vs_Q, ax=axs[4,0], cmap='rainbow', cmin=0.0000001)
    hep.hist2dplot(hdx_vs_Q_n, ax=axs[4,1], cmap='rainbow', cmin=0.0000001)
    
    def get_xpos(frac, ax):
        x0, x1 = ax.get_xlim()
        return frac*(x1-x0) + x0
    def get_ypos(frac, ax):
        y0, y1 = ax.get_ylim()
        return frac*(y1-y0) + y0
    axs[0,0].text(get_xpos(0.7, axs[0,0]), get_ypos(0.8, axs[0,0]), 'mean: {:.4f}'.format(np.mean(hits['dx'])))
    axs[0,0].text(get_xpos(0.7, axs[0,0]), get_ypos(0.7, axs[0,0]), 'stddev: {:.4f}'.format(np.std(hits['dx'], ddof=1)))
    axs[0,0].text(get_xpos(0.7, axs[0,0]), get_ypos(0.6, axs[0,0]), 'skew: {:.4f}'.format(scipy.stats.skew(hits['dx'])))
    
    axs[0,1].text(get_xpos(0.7, axs[0,1]), get_ypos(0.8, axs[0,1]), 'mean: {:.4f}'.format(np.mean(hits['d_nm'])))
    axs[0,1].text(get_xpos(0.7, axs[0,1]), get_ypos(0.7, axs[0,1]), 'stddev: {:.4f}'.format(np.std(hits['d_nm'], ddof=1)))
    axs[0,1].text(get_xpos(0.7, axs[0,1]), get_ypos(0.6, axs[0,1]), 'skew: {:.4f}'.format(scipy.stats.skew(hits['d_nm'])))
    
    return hdx, hyz_perp, hdx_vs_totQ, hdx_vs_totN, hdx_vs_Q, haccQ_vs_dx, hfrac_avg_i_vs_dx, havg_i_vs_dx, fig

In [48]:
def run_on_single_file(f_name = '/pscratch/sd/y/yousen/rock_mu_sub/output/packet-0050017-2024_07_08_15_13_35_CDT.FLOW.rock_mu.h5', track_idxs=[33]):
    packet_prefix = f_name.split('/')[-1].split('.')[0]
    
    # att = 'Q'
    att = 't_drift'
    
    if not os.path.exists(packet_prefix):
        os.makedirs(packet_prefix)

    tracks, tracks2hits = prepare_tracks(f_name)
    
    hdxs = []
    hyz_perps = []
    haccQ_vs_dxs = []
    havg_i_vs_dxs = []
    hfrac_avg_i_vs_dxs = []
    hQ_vs_dxs = []
    
    trkid = []
    
    # for itrk in [100, 103, 107, 108, 130, 140, 25, 30, 33, 47, 59, 67, 77, 86]:

    for itrk in track_idxs:
        extended_hits, fig = analyze_hits_per_track(tracks=tracks, tracks2hits=tracks2hits, itrk=itrk, att=att)
        if len(extended_hits) < 100:
            continue
        # display(fig)
        hdx, hyz_perp, _, _, hQ_vs_dx, haccQ_vs_dx, hfrac_avg_i_vs_dx, havg_i_vs_dx, fig2 = analysis(extended_hits)
        # display(fig2)
        fig.savefig('{}/projection_view_{}_trk{}.png'.format(packet_prefix, att, itrk))
        fig2.savefig('{}/histogram_{}_trk{}.png'.format(packet_prefix, att, itrk))
        
        hdxs.append(hdx)
        hyz_perps.append(hyz_perp)
        haccQ_vs_dxs.append(haccQ_vs_dx)
        havg_i_vs_dxs.append(havg_i_vs_dx)
        hfrac_avg_i_vs_dxs.append(hfrac_avg_i_vs_dx)
        hQ_vs_dxs.append(hQ_vs_dx)
        trkid.append(itrk)
        
    hyz_perp_all = hyz_perps[0]
    for i in range(1, len(hyz_perps)):
        hyz_perp_all = hyz_perp_all+hyz_perps[i]
    hdx_all = hdxs[0]
    for i in range(1, len(hdxs)):
        hdx_all = hdx_all+hdxs[i]
    haccQ_vs_dx_all = haccQ_vs_dxs[0]
    for i in range(1, len(haccQ_vs_dxs)):
        haccQ_vs_dx_all = haccQ_vs_dx_all+haccQ_vs_dxs[i]
    havg_i_vs_dx_all = havg_i_vs_dxs[0]
    for i in range(1, len(havg_i_vs_dxs)):
        havg_i_vs_dx_all = havg_i_vs_dx_all+havg_i_vs_dxs[i]
    hfrac_avg_i_vs_dx_all = hfrac_avg_i_vs_dxs[0]
    for i in range(1, len(hfrac_avg_i_vs_dxs)):
        hfrac_avg_i_vs_dx_all = hfrac_avg_i_vs_dx_all+hfrac_avg_i_vs_dxs[i]
    fig3, axs = plt.subplots(3, 2, figsize=(5*2, 5*3))
    hep.histplot(hdx_all, ax=axs[0,0])
    hep.histplot(hyz_perp_all, ax=axs[0,1])
    hep.hist2dplot(haccQ_vs_dx_all, ax=axs[1,0], cmap='rainbow', cmin=0.0000001)
    hep.hist2dplot(havg_i_vs_dx_all, ax=axs[2,0], cmap='rainbow', cmin=0.00000000001)
    # display(fig3)
    fig3.savefig('{}/histogram_{}_summary.png'.format(packet_prefix, att, itrk))

    fout = uproot.recreate('{}_hist_summary.root'.format(packet_prefix))
    fout['hdx_all'] = hdx_all
    fout['hyz_perp_all'] = hyz_perp_all
    fout['haccQ_vs_dx_all'] = haccQ_vs_dx_all
    fout['havg_i_vs_dx_all'] = havg_i_vs_dx_all
    fout['hfrac_avg_i_vs_dx_all'] = hfrac_avg_i_vs_dx_all
    for h, itrk in zip(hdxs, trkid):
        fout['{}/hdx_trk{}'.format(packet_prefix, itrk)] = h
    for h, itrk in zip(hyz_perps, trkid):
        fout['{}/hyz_perp_trk{}'.format(packet_prefix, itrk)] = h
    for h, itrk in zip(haccQ_vs_dx, trkid):
        fout['{}/haccQ_vs_dx_trk{}'.format(packet_prefix, itrk)] = h
    for h, itrk in zip(havg_i_vs_dxs, trkid):
        fout['{}/havg_i_vs_dx_trk{}'.format(packet_prefix, itrk)] = h
    # for h, Q
        
    plt.close('all')

In [49]:
plt.close('all')
plt.ioff()
run_on_single_file()

RuntimeError: Can't decrement id ref count (can't close file, there are objects still open)

Exception ignored in: 'h5py._objects.ObjectID.__dealloc__'
Traceback (most recent call last):
  File "h5py/_objects.pyx", line 201, in h5py._objects.ObjectID.__dealloc__
RuntimeError: Can't decrement id ref count (can't close file, there are objects still open)


negative Q []
[] [] [] [] []


In [41]:
plt.close('all')
plt.ioff()
input_meta = {
    '/pscratch/sd/y/yousen/rock_mu_sub/output/packet-0050017-2024_07_08_15_03_34_CDT.FLOW.rock_mu.h5' : [1, 105, 11, 112, 122, 155, 156, 24, 49, 52, 66, ]
}
for k, v in input_meta.items():
    run_on_single_file(k, v)
    plt.close('all')

RuntimeError: Can't decrement id ref count (can't close file, there are objects still open)

Exception ignored in: 'h5py._objects.ObjectID.__dealloc__'
Traceback (most recent call last):
  File "h5py/_objects.pyx", line 201, in h5py._objects.ObjectID.__dealloc__
RuntimeError: Can't decrement id ref count (can't close file, there are objects still open)


negative Q [-1.21011939]
[4.11165795] [2.90153856] [-28.] [4.11165795] [(2897, 21.55591975, 3.32550001, 51.0143013, 1158., 9407394, 2, 9, 4.11165795, 0.13838053, 4.16768055, 0.18774335, 2.90153856, 2, 4.11165795, -28., -1.e+09, 0.27904482)]
negative Q [-2.1701778  -0.05064187]
[30.741908   14.74905096] [28.5717302  14.69840909] [-43. -28.] [20.18128704  5.27113547] [(709319, 39.5850994 , 35.25030136,  6.67430019, 1525., 5785144, 1,  1, 20.18128704, 0.67921438, 90.23855119, -0.17741994, 28.5717302 , 4, 30.741908  , -43., 0.72076025, -0.19598724)
 (709527, 38.05250502, 17.51429939, 48.79729843, 1621., 5785240, 1, 27,  5.27113547, 0.1774035 , 44.53394198, -0.21542052, 14.69840909, 4, 14.74905096, -28., 0.0516778 , -0.12872043)]
negative Q []
[] [] [] [] []
negative Q [-1.22787311 -0.37121771]
[] [] [] [] []
negative Q [-8.77188183 -1.29466465]
[11.45087208 14.4403582 ] [ 2.67899025 13.14569356] [-28. -28.] [11.45087208  9.34889004] [(907949, 6.24594045,  -9.08969975, 16.42910004, 199., 17

/tmp/ipykernel_555939/120645153.py:39: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axs = plt.subplots(3, 2, figsize=(5*2, 5*3))
